In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import pandas as pd

df = pd.read_csv("../data/earthquakes_cleaned.csv")

connection_url = URL.create(
    "mysql+pymysql",
    username="your_username",
    password="your_password",
    host="localhost",
    port="your_port",
    database="global_seismic_trends"
)

engine = create_engine(connection_url)

with engine.connect() as connection:
    print("Database connection successful!")

Database connection successful!


In [12]:
df.shape

(102437, 33)

In [13]:
df.columns

Index(['id', 'time', 'updated', 'latitude', 'longitude', 'depth_km', 'mag',
       'magType', 'place', 'status', 'tsunami', 'sig', 'net', 'nst', 'dmin',
       'rms', 'gap', 'type', 'alert', 'felt', 'cdi', 'mmi', 'code', 'types',
       'ids', 'sources', 'country', 'year', 'month', 'day', 'day_of_week',
       'depth_category', 'magnitude_category'],
      dtype='str')

In [14]:
#inserting the dataframe to SQL
df.to_sql(
    "earthquakes",
    con=engine,
    if_exists="replace",
    index=False
)

102437

In [15]:
from sqlalchemy import text
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM earthquakes"))
    print(result.fetchone()[0])

102437


In [16]:
#making sure that the columns in MySQL matchs the DataFrame.
with engine.connect() as conn:
    result = conn.execute(text("DESCRIBE earthquakes"))
    
    for row in result:
        print(row)

('id', 'text', 'YES', '', None, '')
('time', 'text', 'YES', '', None, '')
('updated', 'text', 'YES', '', None, '')
('latitude', 'double', 'YES', '', None, '')
('longitude', 'double', 'YES', '', None, '')
('depth_km', 'double', 'YES', '', None, '')
('mag', 'double', 'YES', '', None, '')
('magType', 'text', 'YES', '', None, '')
('place', 'text', 'YES', '', None, '')
('status', 'text', 'YES', '', None, '')
('tsunami', 'bigint', 'YES', '', None, '')
('sig', 'bigint', 'YES', '', None, '')
('net', 'text', 'YES', '', None, '')
('nst', 'double', 'YES', '', None, '')
('dmin', 'double', 'YES', '', None, '')
('rms', 'double', 'YES', '', None, '')
('gap', 'double', 'YES', '', None, '')
('type', 'text', 'YES', '', None, '')
('alert', 'text', 'YES', '', None, '')
('felt', 'double', 'YES', '', None, '')
('cdi', 'double', 'YES', '', None, '')
('mmi', 'double', 'YES', '', None, '')
('code', 'text', 'YES', '', None, '')
('types', 'text', 'YES', '', None, '')
('ids', 'text', 'YES', '', None, '')
('source

In [17]:
# Verifying the values are stored properly
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT *
        FROM earthquakes
        LIMIT 5
    """))

    for row in result:
        print(row)

('us7000fqcd', '2021-10-31 23:47:49.663', '2022-01-14 17:58:48.040', -17.9423, -178.4634, 570.45, 4.4, 'mb', '235 km E of Levuka, Fiji', 'reviewed', 0, 298, 'us', 32.0, 3.628, 0.69, 121.0, 'earthquake', 'none', 0.0, 0.0, 0.0, '7000fqcd', 'origin,phase-data', ',us7000fqcd,', ',us,', 'Fiji', 2021, 10, 31, 'Sunday', 'deep', 'strong')
('us7000fqca', '2021-10-31 23:39:39.089', '2022-01-14 17:58:48.040', -25.3189, 179.1626, 496.53, 4.4, 'mb', 'south of the Fiji Islands', 'reviewed', 0, 298, 'us', 32.0, 4.707, 0.79, 66.0, 'earthquake', 'none', 0.0, 0.0, 0.0, '7000fqca', 'origin,phase-data', ',us7000fqca,', ',us,', 'south of the Fiji Islands', 2021, 10, 31, 'Sunday', 'deep', 'strong')
('us7000fqce', '2021-10-31 23:27:25.244', '2022-01-14 17:58:48.040', 40.6211, 141.7102, 90.56, 4.2, 'mb', '22 km NE of Hachinohe, Japan', 'reviewed', 0, 271, 'us', 32.0, 0.518, 0.65, 130.0, 'earthquake', 'none', 0.0, 0.0, 0.0, '7000fqce', 'origin,phase-data', ',us7000fqce,', ',us,', 'Japan', 2021, 10, 31, 'Sunday

### Magnitude & Depth

In [18]:
#1. Top 10 strongest earthquakes (mag).
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT id, year, mag, country, depth_km
        FROM earthquakes
        ORDER BY mag DESC
        LIMIT 10;
    """, engine)

result

,id,year,mag,country,depth_km
0,us6000qw60,2025,8.8,Russia Earthquake,35.000
1,us6000tkt2,2026,7.8,Indonesia,10.000
2,us7000srb1,2026,7.8,Philippines,56.991
3,us7000qx2g,2025,7.8,Russia,27.000
4,us6000jllz,2023,7.8,Kahramanmaras earthquake sequence,10.000
5,us6000kd0n,2023,7.7,southeast of the Loyalty Islands,18.053
6,us7000pn9s,2025,7.7,Burma (Myanmar) Earthquake,10.000
7,us7000pcdl,2025,7.6,Cayman Islands,14.326
8,us6000kawn,2023,7.6,Tonga,210.000
9,us7000j36j,2023,7.6,Indonesia,104.950


In [19]:
#2. Top 10 deepest earthquakes (depth_km).
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT id, time, depth_km ,country, mag
        FROM earthquakes
        ORDER BY depth_km DESC
        LIMIT 10;
    """, engine)

result

,id,time,depth_km,country,mag
0,us6000sp3p,2026-04-02 16:59:17.870,683.578,Vanuatu,4.0
1,us6000k2db,2023-04-01 18:09:17.114,681.238,Vanuatu,4.0
2,us7000kxdn,2023-09-18 15:35:27.270,675.265,Vanuatu region,4.2
3,us6000mivr,2024-03-08 22:42:23.713,671.043,Fiji region,4.2
4,us6000rk66,2025-10-29 17:07:34.157,669.556,Fiji,4.8
5,us7000q1jk,2025-05-10 02:03:16.756,667.237,Fiji,4.2
6,us7000s0st,2026-02-14 08:27:48.418,667.197,Fiji,4.3
7,us6000ta9m,2026-07-05 14:22:25.070,665.993,Fiji,5.7
8,us6000tknp,2026-08-10 16:50:09.877,665.326,Vanuatu,4.0
9,us7000he64,2022-06-01 18:41:01.389,664.700,Fiji,4.3


In [20]:
#3. Shallow earthquakes < 50 km and mag > 7.5
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT id, time, depth_km , mag, country
        FROM earthquakes
        WHERE depth_km < 50 AND mag > 7.5;
    """, engine)

result

,id,time,depth_km,mag,country
0,us7000i9bw,2022-09-19 18:05:08.217,26.943,7.6,Mexico
1,us6000jllz,2023-02-06 01:17:34.342,10.000,7.8,Kahramanmaras earthquake sequence
2,us6000kd0n,2023-05-19 02:57:03.172,18.053,7.7,southeast of the Loyalty Islands
3,us7000lff4,2023-12-02 14:37:04.454,40.000,7.6,Philippines
4,us7000pcdl,2025-02-08 23:23:14.697,14.326,7.6,Cayman Islands
5,us7000pn9s,2025-03-28 06:20:52.715,10.000,7.7,Burma (Myanmar) Earthquake
6,us6000qw60,2025-07-29 23:24:52.483,35.000,8.8,Russia Earthquake
7,us7000qx2g,2025-09-18 18:58:14.939,27.000,7.8,Russia
8,us6000rgf4,2025-10-10 20:29:20.075,5.639,7.6,Drake Passage
9,us6000rtdt,2025-12-08 14:15:09.896,40.720,7.6,Japan Earthquake


In [21]:
#5. Average magnitude per magnitude type (magType)
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT magType, avg(mag) as Average_magnitude
        FROM earthquakes
        GROUP BY magType;
    """, engine)

result

,magType,Average_magnitude
0,mb,4.416183
1,ml,3.268790
2,mww,5.354177
3,mwr,4.316355
4,md,3.445154
5,mb_lg,3.196040
6,mw,3.919373
7,mwb,5.765385
8,mlg,3.300000
9,mlr,3.487500


In [22]:
#6. Year with most earthquakes.
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT year, COUNT(*) AS earthquake_count
        FROM earthquakes
        GROUP BY year
        ORDER BY earthquake_count DESC
        LIMIT 1;
    """, engine)

result

,year,earthquake_count
0,2025,22819


In [23]:
#7. Month with highest number of earthquakes.
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT month, COUNT(*) AS earthquake_count
        FROM earthquakes
        GROUP BY month
        ORDER BY earthquake_count DESC
        Limit 1;
    """, engine)

result

,month,earthquake_count
0,7,9990


In [24]:
#8. Day of week with most earthquakes.
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT day_of_week, COUNT(*) AS earthquake_count
        FROM earthquakes
        GROUP BY day_of_week
        ORDER BY earthquake_count DESC
        LIMIT 1
    """, engine)
result


,day_of_week,earthquake_count
0,Tuesday,14900


In [25]:
#9. Count of earthquakes per hour of day.
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT
            HOUR(time) AS hour_of_day,
            COUNT(*) AS earthquake_count
        FROM earthquakes
        GROUP BY HOUR(time)
        ORDER BY hour_of_day;
    """, engine)
result

,hour_of_day,earthquake_count
0,0,4242
1,1,4504
2,2,4367
3,3,4515
4,4,4489
5,5,4210
6,6,4158
7,7,4257
8,8,4293
9,9,4242


In [26]:
#10. Most active reporting network (net).
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT net, COUNT(*) AS earthquake_count
        FROM earthquakes
        GROUP BY net
        ORDER BY earthquake_count DESC
        LIMIT 1
    """, engine)
result

,net,earthquake_count
0,us,89892


In [27]:
#14.  Count of reviewed vs automatic earthquakes (status).
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT status, COUNT(*) AS earthquake_count
        FROM earthquakes
        GROUP BY status
    """, engine)
result


,status,earthquake_count
0,reviewed,102317
1,automatic,120


In [28]:
#15.  Count by earthquake type (type).
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT type, COUNT(*) AS earthquake_count
        FROM earthquakes
        GROUP BY type
    """, engine)
result

,type,earthquake_count
0,earthquake,101669
1,mining explosion,731
2,quarry blast,2
3,volcanic eruption,12
4,ice quake,8
5,landslide,6
6,mine collapse,1
7,other event,5
8,explosion,1
9,experimental explosion,2


In [29]:
#16.  Number of earthquakes by data type (types).
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT types, COUNT(*) AS earthquake_count
        FROM earthquakes
        GROUP BY types
    """, engine)
result

,types,earthquake_count
0,"origin,phase-data",77771
1,"dyfi,origin,phase-data",7301
2,"dyfi,moment-tensor,origin,phase-data",1190
3,"dyfi,earthquake-name,moment-tensor,origin,phas...",38
4,"earthquake-name,origin,phase-data",2103
...,...,...
540,"dyfi,ground-failure,impact-text,internal-momen...",1
541,"dyfi,event-sequence,finite-fault,general-text,...",2
542,"dyfi,event-sequence,moment-tensor,oaf,origin,p...",1
543,"dyfi,event-sequence,impact-link,moment-tensor,...",1


In [30]:
#18 Events with high station coverage
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT id, time, country, mag, nst
        FROM earthquakes
        WHERE nst > 100
        ORDER BY nst DESC;
    """, conn)

result

,id,time,country,mag,nst
0,us6000m12f,2024-01-02 01:17:31.568,Japan,5.4,619.0
1,us6000qzfl,2025-08-09 08:01:41.017,Russia,5.0,566.0
2,us7000rluk,2026-01-01 06:46:54.742,Alaska,5.7,516.0
3,us7000pvtr,2025-04-29 14:53:37.897,Macquarie Island region,6.8,475.0
4,usd001097k,2023-12-11 18:35:59.877,Argentina,5.5,466.0
...,...,...,...,...,...
7888,us6000rdnq,2025-09-29 22:07:23.711,Russia,5.5,101.0
7889,us6000rcd9,2025-09-23 17:15:21.996,Russia,4.6,101.0
7890,us7000qvc9,2025-09-10 20:39:03.893,Mexico,4.4,101.0
7891,us7000quy7,2025-09-09 06:56:23.768,Philippines,4.7,101.0


In [31]:
#19.  Number of tsunamis triggered per year.
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT year, COUNT(*) AS No_of_tsunamis
        FROM earthquakes WHERE tsunami = 1
        GROUP BY year
    """, engine)
result

,year,No_of_tsunamis
0,2021,31
1,2022,136
2,2023,119
3,2024,114
4,2025,142
5,2026,41


In [32]:
#20.  Count earthquakes by alert levels (red, orange, etc.).
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT alert as Alert_Levels, COUNT(*) AS Count_of_Earthquakes
        FROM earthquakes
        GROUP BY alert
    """, engine)
result

,Alert_Levels,Count_of_Earthquakes
0,none,98368
1,green,3901
2,yellow,122
3,red,23
4,orange,23


In [33]:
#21 Find the top 5 countries with the highest average magnitude of earthquakes in the past 5 years  
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT country, AVG(mag) AS avg_magnitude
        FROM earthquakes
        GROUP BY country
        ORDER BY avg_magnitude DESC
        LIMIT 5;
    """, engine)
result

,country,avg_magnitude
0,Russia Earthquake,8.10
1,Burma (Myanmar) Earthquake,7.70
2,2025 Southern Drake Passage Earthquake,7.50
3,2025 Drake Passage Earthquake,7.40
4,Japan Earthquake,7.25


In [34]:
#22. Find countries that have experienced both shallow and deep earthquakes within the same month.
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT country, month
        FROM earthquakes
        WHERE country IS NOT NULL
        GROUP BY country, month
        HAVING COUNT(DISTINCT depth_category) = 2
        LIMIT 20;
    """, engine)
result

,country,month
0,Afghanistan,1
1,Afghanistan,2
2,Afghanistan,3
3,Afghanistan,4
4,Afghanistan,5
5,Afghanistan,6
6,Afghanistan,7
7,Afghanistan,8
8,Afghanistan,9
9,Afghanistan,10


In [35]:
#23. Compute the year-over-year growth rate in the total number of earthquakes globally
with engine.connect() as conn:
    result = pd.read_sql("""
        WITH yearly_counts AS (
            SELECT year, COUNT(*) AS earthquake_count
            FROM earthquakes
            GROUP BY year
        )

        SELECT year, earthquake_count,
            LAG(earthquake_count) OVER (ORDER BY year) AS previous_year_count,
            ROUND(
                (earthquake_count - LAG(earthquake_count) OVER (ORDER BY year))
                / LAG(earthquake_count) OVER (ORDER BY year) * 100,
                2
            ) AS yoy_growth_rate
        FROM yearly_counts
        ORDER BY year;
    """, conn)

result

,year,earthquake_count,previous_year_count,yoy_growth_rate
0,2021,4994,NaN,NaN
1,2022,20208,4994.0,304.65
2,2023,20830,20208.0,3.08
3,2024,18659,20830.0,-10.42
4,2025,22819,18659.0,22.29
5,2026,14927,22819.0,-34.59


In [36]:
#24.  List the 3 most seismically active regions by combining both frequency and average magnitude.
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT country, COUNT(*) AS earthquake_count, ROUND(AVG(mag),2) AS avg_magnitude
        FROM earthquakes
        GROUP BY country
        ORDER BY earthquake_count DESC, avg_magnitude DESC
        LIMIT 3;
    """, conn)

result

,country,earthquake_count,avg_magnitude
0,Alaska,12065,3.47
1,Indonesia,8215,4.50
2,Russia,6112,4.48


In [37]:
#25. For each country, calculate the average depth of earthquakes within ±5° latitude range of the equator.
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT country, ROUND(AVG(depth_km),2) AS average_depth
        FROM earthquakes
        WHERE latitude BETWEEN -5 and +5
        GROUP BY country   
    """, conn)
result

,country,average_depth
0,Philippines,105.64
1,Indonesia,61.59
2,north of Ascension Island,10.00
3,central Mid-Atlantic Ridge,10.01
4,Colombia,48.26
5,Papua New Guinea,72.45
6,central East Pacific Rise,10.00
7,Federated States of Micronesia region,10.00
8,Uganda,10.08
9,"Laikit II (Dimembe), Indonesia",85.43


In [38]:
# 26. Identify countries having the highest ratio of shallow to deep earthquakes.
with engine.connect() as conn:
    result = pd.read_sql("""
        SELECT  country,
        SUM(CASE WHEN depth_category = 'shallow' THEN 1 ELSE 0 END) AS shallow_count,
        SUM(CASE WHEN depth_category = 'deep' THEN 1 ELSE 0 END) AS deep_count,
        ROUND(
            SUM(CASE WHEN depth_category = 'shallow' THEN 1 ELSE 0 END)
            /
            NULLIF(SUM(CASE WHEN depth_category = 'deep' THEN 1 ELSE 0 END), 0),
            2
        ) AS shallow_to_deep_ratio
        FROM earthquakes
        WHERE country IS NOT NULL
        GROUP BY country
        HAVING deep_count > 0
        ORDER BY shallow_to_deep_ratio DESC
        LIMIT 20;
        """, conn)
result

,country,shallow_count,deep_count,shallow_to_deep_ratio
0,Canada,269.0,1.0,269.00
1,Iran,862.0,4.0,215.50
2,Panama,187.0,2.0,93.50
3,China,1260.0,17.0,74.12
4,Nepal,145.0,2.0,72.50
5,Pakistan,216.0,3.0,72.00
6,Turkey,926.0,15.0,61.73
7,Morocco,48.0,1.0,48.00
8,Cyprus,34.0,1.0,34.00
9,Antigua and Barbuda,31.0,1.0,31.00


In [39]:
#27. Find the average magnitude difference between earthquakes with tsunami alerts and those without.
with engine.connect() as conn:
    result = pd.read_sql("""
    SELECT
    ROUND(AVG(CASE WHEN tsunami = 1 THEN mag END), 2) AS avg_mag_with_tsunami,
    ROUND(AVG(CASE WHEN tsunami = 0 THEN mag END), 2) AS avg_mag_without_tsunami,
    ROUND(
        AVG(CASE WHEN tsunami = 1 THEN mag END) -
        AVG(CASE WHEN tsunami = 0 THEN mag END),
        2
    ) AS average_magnitude_difference
    FROM earthquakes;
""", conn)
result

,avg_mag_with_tsunami,avg_mag_without_tsunami,average_magnitude_difference
0,5.39,4.24,1.14


In [40]:
# 28. Using the gap and rms columns, identify events with the lowest data reliability (highest average error margins).
with engine.connect() as conn:
    result = pd.read_sql("""
    SELECT
    id,
    country,
    mag,
    depth_km,
    gap,
    rms,
    ROUND((gap + rms) / 2, 2) AS average_error_margin
FROM earthquakes
WHERE gap IS NOT NULL
  AND rms IS NOT NULL
ORDER BY average_error_margin DESC
LIMIT 10;
""", conn)
result

,id,country,mag,depth_km,gap,rms,average_error_margin
0,pr71519528,U.S. Virgin Islands,3.31,62.920,359.00,0.1400,179.57
1,nn00897599,Nevada,3.00,0.000,358.18,0.1601,179.17
2,pr2024051000,Anguilla,3.58,30.000,358.00,0.1600,179.08
3,pr71508583,Anguilla,3.17,42.820,356.00,0.2200,178.11
4,aka2026fwzivr,south of Alaska,3.90,5.000,354.00,0.7000,177.35
5,pr2022227000,U.S. Virgin Islands,3.63,14.000,353.00,0.6100,176.80
6,pr71489653,Dominican Republic,3.13,46.940,352.00,0.1800,176.09
7,pr2022289000,U.S. Virgin Islands,3.48,44.000,352.00,0.0700,176.04
8,pr2022311006,U.S. Virgin Islands,3.13,52.000,351.00,0.1700,175.58
9,us6000k2l1,Alaska,3.00,155.111,350.00,0.2400,175.12


In [41]:
#30 Determine the regions with the highest frequency of deep-focus earthquakes (depth > 300 km).
with engine.connect() as conn:
    result = pd.read_sql("""
    SELECT
    country,
    COUNT(*) AS deep_focus_earthquakes
    FROM earthquakes
    WHERE depth_km > 300
    AND country IS NOT NULL
    GROUP BY country
    ORDER BY deep_focus_earthquakes DESC
    LIMIT 10;
""", conn)
result

,country,deep_focus_earthquakes
0,south of the Fiji Islands,1509
1,Fiji region,1114
2,Fiji,1068
3,Tonga,545
4,Indonesia,251
5,Japan region,225
6,Timor Leste,190
7,Kermadec Islands region,147
8,Wallis and Futuna,136
9,Japan,119
